In [ ]:
p_keyword_single = "" 
p_folder_path = ""

In [ ]:
path = f"Files/{p_folder_path}/youtube_{p_keyword_single}_*.json"
df_raw = spark.read.option('multiline','true').json(path)
df_raw.printSchema()

In [ ]:
# 2. Xử lý Videos thường
from pyspark.sql.functions import explode, col, lit
df_videos = df_raw.select(   #explode mảng videos + cột q mảng search_parameter + cột created_at mảng search_metadata
                            explode(col("videos")).alias("v"),
                            col("search_parameters.q").alias("keyword"),
                            col("search_metadata.created_at").alias("fetched_at")
                                ).select(
                                    col("v.id").alias("video_id"),
                                    col("v.title"),
                                    col("v.channel.title").alias("channel_name"),
                                    col("v.views").alias("view_count"),
                                    col("v.published_time"),
                                    col("v.length").alias("duration"),
                                    col("keyword"),
                                    col("fetched_at"),
                                    lit("Regular").alias("video_type") #thêm cột để phân biệt với mảng shorts
                                )

In [5]:
# 3. Xử lý phần Shorts
from pyspark.sql.types import StructType, StructField, StringType, LongType, TimestampType
# Kiểm tra xem cột 'shorts' có tồn tại trong file JSON
if "shorts" in df_raw.columns:
    # Nếu có thì chạy logic
    df_shorts = df_raw.select(
                            explode(col("shorts")).alias("s_group"),
                            col("search_parameters.q").alias("keyword"),
                            col("search_metadata.created_at").alias("fetched_at"))\
                                .select(
                                    explode(col("s_group.items")).alias("s"),
                                    col("keyword"),
                                    col("fetched_at"))\
                                        .select(
                                            col("s.id").alias("video_id"),
                                            col("s.title"),
                                            lit("Unknown Artist").alias("channel_name"),
                                            col("s.views").alias("view_count"),
                                            lit(None).alias("published_time"),
                                            lit(None).alias("duration"),
                                            col("keyword"),
                                            col("fetched_at"),
                                            lit("Shorts").alias("video_type"))
else:
    # Nếu không CÓ cột shorts, ta tạo một DataFrame rỗng với đúng cấu trúc để union
    print(f"Keyword '{p_keyword_single}' không có Shorts. Tạo bảng rỗng.")
    
    # Tạo schema khớp với df_videos để khi Union
    empty_schema = StructType([
                                StructField("video_id", StringType(), True),
                                StructField("title", StringType(), True),
                                StructField("channel_name", StringType(), True),
                                StructField("view_count", StringType(), True),
                                StructField("published_time", StringType(), True),
                                StructField("duration", StringType(), True),
                                StructField("keyword", StringType(), True),
                                StructField("fetched_at", StringType(), True),
                                StructField("video_type", StringType(), True)
                            ])
    df_shorts = spark.createDataFrame([], empty_schema)

StatementMeta(, d4a3da32-1b90-4aa4-a5b0-a04dd64f4a4d, 7, Finished, Available, Finished, False)

In [7]:
# Gộp các bảng lại
df_silver = df_videos.unionByName(df_shorts, allowMissingColumns=True)

StatementMeta(, d4a3da32-1b90-4aa4-a5b0-a04dd64f4a4d, 9, Finished, Available, Finished, False)

In [9]:
from delta.tables import DeltaTable
from pyspark.sql.functions import trim, when, col, to_date

# 1. Transform Silver
df_updates = df_silver.select(
                                col("video_id"),
                                trim(col("title")).alias("title"),
                                col("channel_name"),
                                col("view_count").cast("long"),
                                col("published_time"),
                                col("duration"),
                                col("keyword"),
                                col("fetched_at").cast("timestamp"),
                                col("video_type"))\
                        .withColumn("channel_name", when(col("channel_name").isNull(), "Unknown Artist").otherwise(col("channel_name")))\
                        .withColumn('date', to_date(col('fetched_at')))\
                        .dropDuplicates(['video_id'])

# 2. Định nghĩa tên bảng đích
target_table_name = "silver_youtube_final"

# 3. Kiểm tra và Merge
if spark.catalog.tableExists(target_table_name):
    print(f"Đang Upsert dữ liệu cho keyword: {p_keyword_single}") 
    target_delta_table = DeltaTable.forName(spark, target_table_name)   
    target_delta_table.alias("target") \
                        .merge(df_updates.alias("updates"),"target.video_id = updates.video_id") \
                        .whenMatchedUpdateAll() \
                        .whenNotMatchedInsertAll() \
                        .execute()
else:
    print("Bảng chưa tồn tại, đang tạo mới")
    df_updates.write.format("delta").mode("overwrite").saveAsTable(target_table_name)

StatementMeta(, d4a3da32-1b90-4aa4-a5b0-a04dd64f4a4d, 11, Finished, Available, Finished, False)